In [61]:
import importlib
import numpy as np
import polars as pl
import scipy.sparse as sp
import torch
import torch.nn as nn
from tqdm import tqdm

from datasets import DATA_FOLDER, Dataloader, prepare_interaction_data
from util import CHECKPOINT_FOLDER, get_checkpoint_filepath, load_checkpoint, load_config_from_checkpoint

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.mps.is_available() else torch.device("cpu")

DATASET = "ML-25M"
# SAE_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TopKSAE-8192-3c29e9ee.ckpt"  # ELSA + Cosine
SAE_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TopKSAE-8192-d6337b64.ckpt"  # ELSA + L2
# SAE_CHECKPOINT_PATH = f"{CHECKPOINT_FOLDER}/{DATASET}/TopKSAE-8192-f634db83.ckpt"  # MultVAE + L2

sae_cfg = load_config_from_checkpoint(SAE_CHECKPOINT_PATH)
elsa_checkpoint_path = f"{CHECKPOINT_FOLDER}/{DATASET}/{sae_cfg['pretrained_model_checkpoint']}"
elsa_cfg = load_config_from_checkpoint(elsa_checkpoint_path)

interactions_df, train_csr, val_csr, test_csr, train_users, val_users, test_users, items = prepare_interaction_data(elsa_cfg)
train_users_to_idxs = {uid: uidx for uidx, uid in enumerate(train_users)}
val_users_to_idxs = {uid: uidx for uidx, uid in enumerate(val_users)}
test_users_to_idxs = {uid: uidx for uidx, uid in enumerate(test_users)}
items_to_idxs = {iid: iidx for iidx, iid in enumerate(items)}

items_df = (
    pl.scan_csv(f"{DATA_FOLDER}/{DATASET}/movies.csv").rename({"movieId": "item_id"}).cast({"item_id": pl.String}).cast({"item_id": pl.Categorical}).collect()
)

elsa_model_class = getattr(importlib.import_module(elsa_cfg["model_module"]), elsa_cfg["model_class"])
elsa = elsa_model_class(train_csr.shape[1], elsa_cfg["embedding_dim"]).to(device)
_, _ = load_checkpoint(elsa, None, get_checkpoint_filepath(elsa_cfg), device, None)

sae_model_class = getattr(importlib.import_module(sae_cfg["model_module"]), sae_cfg["model_class"])
sae_extra_params = {k: sae_cfg[k] for k in sae_cfg.keys() if k in ["l1_coef", "k"]}
sae = sae_model_class(elsa_cfg["embedding_dim"], sae_cfg["embedding_dim"], sae_cfg["reconstruction_loss"], l1_coef=sae_cfg["l1_coef"], k=sae_cfg["k"]).to(
    device
)
_, _ = load_checkpoint(sae, None, get_checkpoint_filepath(sae_cfg), device, None)

Removing users with < 5 interactions...
Dataset info: users=160776, items=40857, interactions=12448242
Train split info: users=128621, items=40857, interactions=9960673
Val split info: users=16078, items=40857, interactions=1253135
Test split info: users=16077, items=40857, interactions=1234434
Loaded checkpoint from checkpoints/ML-25M/ELSA-1024-10977915.ckpt (after 13 epochs)
Loaded checkpoint from checkpoints/ML-25M/TopKSAE-8192-d6337b64.ckpt (after 761 epochs)


In [62]:
class ELSAWithSAE:
    def __init__(self, elsa, sae):
        self.elsa = elsa
        self.sae = sae

    @torch.no_grad()
    def recommend_without_sae(self, interaction_batch: torch.Tensor, k: int, mask_interactions: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
        return self.elsa.recommend(interaction_batch, k, mask_interactions)

    @torch.no_grad()
    def encode(self, interaction_batch: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        elsa_embeddings = self.elsa.encode(interaction_batch)
        sae_embeddings, _, input_mean, input_std = sae.encode(elsa_embeddings)
        return sae_embeddings, input_mean, input_std

    @torch.no_grad()
    def decode(self, sparse_activations: torch.Tensor, input_mean: torch.Tensor, input_std: torch.Tensor) -> torch.Tensor:
        return self.elsa.decode(self.sae.decode(sparse_activations, input_mean, input_std))

    @torch.no_grad()
    def recommend_with_sae(
        self, interaction_batch: torch.Tensor, k: int, mask_interactions: bool = True, activation_boost: dict[tuple[int], float] | None = None
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        sparse_activations, input_mean, input_std = self.encode(interaction_batch)
        if activation_boost is not None:
            s = sparse_activations.sum(-1, keepdim=True)
            sparse_activations /= s  # make all activations sum to 1
            boost_volume = np.sum([v for v in activation_boost.values()])  # sum must be between 0 and 1
            sparse_activations *= 1 - boost_volume
            for idx, boost_value in activation_boost.items():
                sparse_activations[*idx] += boost_value
            sparse_activations *= s  # rescale back
        scores = nn.ReLU()(self.decode(sparse_activations, input_mean, input_std) - interaction_batch)
        if mask_interactions:
            scores = torch.where(interaction_batch != 0, 0, scores)  # mask input interactions
        topk_scores, topk_indices = torch.topk(scores, k)
        return topk_scores.cpu().numpy(), topk_indices.cpu().numpy(), sparse_activations.cpu().numpy()


elsa_with_sae = ELSAWithSAE(elsa, sae)

In [63]:
sparse_embeddings = []
onehot_items_dataloader = Dataloader(sp.eye(len(items), dtype=np.float32, format="csr"), batch_size=1024, device=device)
with torch.no_grad():
    for onehot_batch in tqdm(onehot_items_dataloader):
        elsa_embedding = elsa.encode(onehot_batch)  # Tensor, shape = (batch.shape[0] x elsa_cfg["embedding_dim"])
        sae_embedding, _, input_mean, input_std = sae.encode(elsa_embedding)
        sparse_embeddings.append(sp.csr_matrix(sae_embedding.cpu().numpy()))
sparse_embeddings = sp.vstack(sparse_embeddings)

neuron_is_alive = np.asarray(sparse_embeddings.sum(axis=0)).flatten() != 0
living_neurons = np.where(neuron_is_alive)[0]
dead_neuron_count = sparse_embeddings.shape[1] - neuron_is_alive.sum()
print(f"{dead_neuron_count} dead neurons out of {sparse_embeddings.shape[1]} ({dead_neuron_count / sparse_embeddings.shape[1]:.2%})")
print(f"{neuron_is_alive.sum()} alive ones")

100%|██████████| 40/40 [00:05<00:00,  6.86it/s]

4267 dead neurons out of 8192 (52.09%)
3925 alive ones


In [64]:
tag_df = (
    pl.scan_csv(f"{DATA_FOLDER}/{DATASET}/tags.csv")
    .rename({"userId": "user_id", "movieId": "item_id"})
    .cast({"user_id": pl.String, "item_id": pl.String})
    .cast({"user_id": pl.Categorical, "item_id": pl.Categorical})
    .with_columns(pl.col("tag").str.to_lowercase().str.strip_chars().alias("tag"))  # convert to lowercase and strip whitespace
    .filter(pl.col("tag").count().over("tag") >= 100)  # keep only tags assigned at least 100 times
    .filter(pl.col("item_id").is_in(items))  # keep only items with interactions
    .with_columns(pl.col("item_id").replace_strict(items_to_idxs).alias("item_idx"))
    .cast({"tag": pl.Categorical})
    .collect()
)

tag_df

user_id,item_id,tag,timestamp,item_idx
cat,cat,cat,i64,i64
"""3""","""260""","""classic""",1439472355,43
"""3""","""260""","""sci-fi""",1439472256,43
"""4""","""1732""","""dark comedy""",1573943598,181
"""4""","""1732""","""great dialogue""",1573943604,181
"""4""","""7569""","""so bad it's good""",1573943455,5010
…,…,…,…,…
"""162462""","""260""","""space""",1427470029,43
"""162492""","""260""","""classic sci-fi""",1436468895,43
"""162492""","""260""","""epic""",1436468882,43


In [65]:
tags = tag_df["tag"].unique(maintain_order=True).to_numpy()
tag_item_counts = sp.csr_matrix(
    (
        np.ones(len(tag_df), dtype=np.float32),
        (
            tag_df["tag"].to_physical().to_numpy(),
            tag_df["item_idx"].to_numpy(),
        ),
    ),
    shape=(tag_df["tag"].n_unique(), len(items)),
)

tag_item_counts

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 209685 stored elements and shape (1633, 40857)>

In [66]:
pl.Series(tag_item_counts.data).describe()

statistic,value
str,f64
"""count""",209685.0
"""null_count""",0.0
"""mean""",3.42385
"""std""",8.698483
"""min""",1.0
"""25%""",1.0
"""50%""",1.0
"""75%""",2.0
"""max""",658.0


In [131]:
def compute_tfidf(X):
    """
    Compute TF-IDF for a term-document value matrix.
    Parameters:
    X: np.ndarray (num_terms, num_documents)
    Returns:
    tfidf_matrix: np.ndarray (num_terms, num_documents) - TF-IDF values
    """
    # Compute Term Frequency (TF) - Normalize by column sum
    tf = X / np.sum(X, axis=0, keepdims=True)
    tf[np.isnan(tf)] = 0  # Handle division by zero
    # Compute Document Frequency (DF) - Count nonzero occurrences of each term
    df = np.count_nonzero(X, axis=1)
    # Compute Inverse Document Frequency (IDF) - Log-scaled
    num_documents = X.shape[1]
    idf = np.log((num_documents + 1) / (df + 1)) + 1  # Smoothing
    # Compute TF-IDF
    tfidf_matrix = tf * idf[:, np.newaxis]
    return tfidf_matrix


pti = tag_item_counts.copy()
pti.data /= pti.data.sum()
tag_neuron_activity = pti @ sparse_embeddings

# top_tag_per_neuron = tags[compute_tfidf(tag_neuron_activity.toarray()).argmax(axis=0)]  # term = tag, document = neuron
top_tag_per_neuron = tags[compute_tfidf(tag_neuron_activity.toarray().T).argmax(axis=1)]  # term = neuron, document = tag
top_tag_per_neuron[~neuron_is_alive] = None

top_neuron_per_tag = compute_tfidf(tag_neuron_activity.toarray().T).argmax(axis=0)  # term = neuron, document = tag
# top_neuron_per_tag = compute_tfidf(tag_neuron_activity.toarray()).argmax(axis=1)  # term = tag, document = neuron

print(pl.Series("top_neuron", top_neuron_per_tag).value_counts(sort=True))
print(pl.Series("top_tag", top_tag_per_neuron[neuron_is_alive]).value_counts(sort=True))

shape: (674, 2)
┌────────────┬───────┐
│ top_neuron ┆ count │
│ ---        ┆ ---   │
│ i64        ┆ u32   │
╞════════════╪═══════╡
│ 4130       ┆ 24    │
│ 2037       ┆ 19    │
│ 5001       ┆ 14    │
│ 1690       ┆ 13    │
│ 4100       ┆ 13    │
│ …          ┆ …     │
│ 7481       ┆ 1     │
│ 1113       ┆ 1     │
│ 999        ┆ 1     │
│ 8147       ┆ 1     │
│ 1395       ┆ 1     │
└────────────┴───────┘
shape: (985, 2)
┌──────────────────┬───────┐
│ top_tag          ┆ count │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ audrey hepburn   ┆ 19    │
│ aardman          ┆ 18    │
│ woody allen      ┆ 18    │
│ sergio leone     ┆ 17    │
│ monty python     ┆ 16    │
│ …                ┆ …     │
│ prostitute       ┆ 1     │
│ darren aronofsky ┆ 1     │
│ wheelchair       ┆ 1     │
│ kick-butt women  ┆ 1     │
│ government       ┆ 1     │
└──────────────────┴───────┘


In [132]:
tag_neuron_activity

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1663710 stored elements and shape (1633, 8192)>

# Explaining recommendations

In [133]:
k = 5

# user_id = np.random.choice(train_users)
user_id = "500"
print(f"User {user_id}")
print("Interactions")
print(
    interactions_df.filter(pl.col("user_id") == user_id)
    .join(items_df, on="item_id")
    # .select(["item_id", "value", "title", "genres"])
    .select(["title", "genres"])
)
interactions = torch.tensor(train_csr[train_users_to_idxs[user_id]].toarray()).to(device)  # Tensor shape = (1 x num_items)

topk_elsa_scores, topk_elsa_idxs = elsa_with_sae.recommend_without_sae(interactions, k)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.flatten(), topk_elsa_idxs.flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations from ELSA:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

topk_scores, topk_idxs, sparse_activations = elsa_with_sae.recommend_with_sae(interactions, k)
topk_scores, topk_idxs, sparse_activations = topk_scores.flatten(), topk_idxs.flatten(), sparse_activations.flatten()
topk_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations from ELSA with SAE:")
print(
    pl.DataFrame({"item_id": topk_item_ids, "score": topk_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

print(f"Active neuron count: {np.count_nonzero(sparse_activations)}")
topk_neuron_values, topk_neuron_ids = torch.topk(torch.tensor(sparse_activations), 10)
print(f"Most active neurons: {top_tag_per_neuron[topk_neuron_ids.cpu().numpy().flatten()]} with values {topk_neuron_values.detach().cpu().numpy().flatten()}")

User 500
Interactions
shape: (11, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ title                           ┆ genres                          │
│ ---                             ┆ ---                             │
│ str                             ┆ str                             │
╞═════════════════════════════════╪═════════════════════════════════╡
│ Casablanca (1942)               ┆ Drama|Romance                   │
│ Apocalypse Now (1979)           ┆ Action|Drama|War                │
│ Annie Hall (1977)               ┆ Comedy|Romance                  │
│ Celebration, The (Festen) (199… ┆ Drama                           │
│ Fight Club (1999)               ┆ Action|Crime|Drama|Thriller     │
│ …                               ┆ …                               │
│ Dancer in the Dark (2000)       ┆ Drama|Musical                   │
│ Shrek (2001)                    ┆ Adventure|Animation|Children|C… │
│ Mulholland Drive (2001)         ┆ Crime|Drama|Film-

/tmp/ipykernel_1776867/1043620219.py:9: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_1776867/1043620219.py:21: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_1776867/1043620219.py:31: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")


In [134]:
k = 6

user_id = np.random.choice(test_users)
# user_id = "152535"
user_id = "157208"
print(f"User {user_id}")
print("Interactions")
print(
    interactions_df.filter(pl.col("user_id") == user_id)
    .join(items_df, on="item_id")
    # .select(["item_id", "value", "title", "genres"])
    .select(["title", "genres"])
)
interactions = torch.tensor(test_csr[test_users_to_idxs[user_id]].toarray()).to(device)  # Tensor shape = (1 x num_items)

topk_elsa_scores, topk_elsa_idxs = elsa_with_sae.recommend_without_sae(interactions, k)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.flatten(), topk_elsa_idxs.flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations from ELSA:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

topk_scores, topk_idxs, sparse_activations = elsa_with_sae.recommend_with_sae(interactions, k)
topk_scores, topk_idxs, sparse_activations = topk_scores.flatten(), topk_idxs.flatten(), sparse_activations.flatten()
topk_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations from ELSA with SAE:")
print(
    pl.DataFrame({"item_id": topk_item_ids, "score": topk_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

print(f"Active neuron count: {np.count_nonzero(sparse_activations)}")
topk_neuron_values, topk_neuron_ids = torch.topk(torch.tensor(sparse_activations), 10)
print(f"Most active neurons: {top_tag_per_neuron[topk_neuron_ids.cpu().numpy().flatten()]} with values {topk_neuron_values.detach().cpu().numpy().flatten()}")

User 157208
Interactions
shape: (6, 2)
┌─────────────────────────────────┬─────────────────────────┐
│ title                           ┆ genres                  │
│ ---                             ┆ ---                     │
│ str                             ┆ str                     │
╞═════════════════════════════════╪═════════════════════════╡
│ Bridges of Madison County, The… ┆ Drama|Romance           │
│ Some Like It Hot (1959)         ┆ Comedy|Crime            │
│ Big Sleep, The (1946)           ┆ Crime|Film-Noir|Mystery │
│ Do You Remember Dolly Bell? (S… ┆ Comedy|Drama|Romance    │
│ Permanent Vacation (1982)       ┆ Drama                   │
│ Navigator, The (1924)           ┆ Comedy                  │
└─────────────────────────────────┴─────────────────────────┘
Recommendations from ELSA:
shape: (6, 4)
┌─────────┬──────────┬────────────────────────────┬──────────────────────────────┐
│ item_id ┆ score    ┆ title                      ┆ genres                       │
│ ---     

/tmp/ipykernel_1776867/2311047409.py:10: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_1776867/2311047409.py:22: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_1776867/2311047409.py:32: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")


# Steering recommendations

In [136]:
pl.DataFrame({"tag": tags, "top neuron": top_neuron_per_tag}).sort(by="tag")

tag,top neuron
str,i64
"""007""",7330
"""01/11""",4130
"""01/12""",1395
"""02/11""",6376
"""03/11""",4130
…,…
"""youtube""",8070
"""zach galifianakis""",3765
"""zombie""",4497


In [116]:
pl.DataFrame({"tag": tags, "top neuron": top_neuron_per_tag}).sort(by="tag").filter(pl.col("tag").str.starts_with("cy"))

tag,top neuron
str,i64
"""cyberpunk""",3171
"""cyborg""",6916
"""cyborgs""",7636
"""cynical""",1792


In [138]:
k = 10
activation_boost = {}

# activation_boost = {(0, 6040): 0.3}  # space action, space adventure
# activation_boost = {(0, 6012): 0.2}  # space, space travel
# activation_boost = {(0, 1991): 0.1}  # star wars
# activation_boost = {(0, 6012): 0.15, (0, 1991): 0.05}  # both
activation_boost = {(0, 7330): 0.15}  # spaghetti western

user_id = np.random.choice(train_users)
user_id = "113638"
# user_id = "80629"
# user_id = "91435"
# user_id = "118099"
print(f"User {user_id}")
print("Interactions")
print(interactions_df.filter(pl.col("user_id") == user_id).join(items_df, on="item_id").select(["item_id", "value", "title", "genres"]).head(10))
interactions = torch.tensor(train_csr[train_users_to_idxs[user_id]].toarray()).to(device)  # Tensor shape = (1 x num_items)
# item_id = "296"
# print(items_df.filter(pl.col("item_id") == item_id).select(["item_id", "title", "genres"]))
# interactions = torch.tensor(sp.eye(len(items), dtype=np.float32, format="csr")[items_to_idxs[item_id]].toarray()).to(device).float()

topk_elsa_scores, topk_elsa_idxs = elsa_with_sae.recommend_without_sae(interactions, k)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.flatten(), topk_elsa_idxs.flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations from ELSA:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
    .head(5)
)

topk_scores, topk_idxs, sparse_activations = elsa_with_sae.recommend_with_sae(interactions, k, activation_boost=activation_boost)
topk_scores, topk_idxs, sparse_activations = topk_scores.flatten(), topk_idxs.flatten(), sparse_activations.flatten()
topk_item_ids = np.array([items[iidx] for iidx in topk_idxs])
print("Recommendations from ELSA with SAE:")
print(
    pl.DataFrame({"item_id": topk_item_ids, "score": topk_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
    .head(10)
)

print(f"Active neuron count: {np.count_nonzero(sparse_activations)}")
topk_neuron_values, topk_neuron_ids = torch.topk(torch.tensor(sparse_activations), 10)
print(f"Most active neurons: {top_tag_per_neuron[topk_neuron_ids.cpu().numpy().flatten()]} with values {topk_neuron_values.detach().cpu().numpy().flatten()}")

User 113638
Interactions
shape: (10, 4)
┌─────────┬───────┬─────────────────────────────────┬─────────────────────────────────┐
│ item_id ┆ value ┆ title                           ┆ genres                          │
│ ---     ┆ ---   ┆ ---                             ┆ ---                             │
│ cat     ┆ f32   ┆ str                             ┆ str                             │
╞═════════╪═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 50      ┆ 4.0   ┆ Usual Suspects, The (1995)      ┆ Crime|Mystery|Thriller          │
│ 199     ┆ 4.0   ┆ Umbrellas of Cherbourg, The (P… ┆ Drama|Musical|Romance           │
│ 541     ┆ 4.0   ┆ Blade Runner (1982)             ┆ Action|Sci-Fi|Thriller          │
│ 899     ┆ 4.0   ┆ Singin' in the Rain (1952)      ┆ Comedy|Musical|Romance          │
│ 900     ┆ 5.0   ┆ American in Paris, An (1951)    ┆ Musical|Romance                 │
│ 901     ┆ 4.0   ┆ Funny Face (1957)               ┆ Comedy|Musical            

/tmp/ipykernel_1776867/3022454587.py:17: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  print(interactions_df.filter(pl.col("user_id") == user_id).join(items_df, on="item_id").select(["item_id", "value", "title", "genres"]).head(10))
/tmp/ipykernel_1776867/3022454587.py:29: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_1776867/3022454587.py:40: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")


In [13]:
pl.DataFrame({"tag": tags, "top neuron": top_neuron_per_tag}).sort(by="tag")[-270:-260]

tag,top neuron
str,i64
"""space""",2356
"""space action""",3125
"""space adventure""",3125
"""space opera""",901
"""space program""",1816
"""space travel""",6695
"""spaceship""",1278
"""spaghetti western""",6073
"""spain""",5046


In [14]:
pl.DataFrame({"tag": tags, "top neuron": top_neuron_per_tag}).sort(by="tag")[-10:]

tag,top neuron
str,i64
"""wry""",6876
"""wwii""",1416
"""x-men""",8184
"""yakuza""",5848
"""youth""",2152
"""youtube""",7350
"""zach galifianakis""",5808
"""zombie""",3696
"""zombies""",358


In [35]:
pl.DataFrame({"tag": tags}).filter(pl.col("tag").str.starts_with("harri"))

tag
str
"""harrison ford"""


In [37]:
top_neuron_per_tag[447]

np.int64(1656)

In [119]:
k = 10

activation_boost = {}
activation_boost = {(0, 1416): 0.5}  # wwii
# activation_boost = {(0, top_neuron_per_tag[447]): 0.4}  # harrison ford

user_id = np.random.choice(train_users)
user_id = "152535"
print(f"User {user_id}")
print("Interactions")
print(interactions_df.filter(pl.col("user_id") == user_id).join(items_df, on="item_id").select(["item_id", "value", "title", "genres"]))
interactions = torch.tensor(test_csr[test_users_to_idxs[user_id]].toarray()).to(device)  # Tensor shape = (1 x num_items)
# item_id = "296"
# print(items_df.filter(pl.col("item_id") == item_id).select(["item_id", "title", "genres"]))
# interactions = torch.tensor(sp.eye(len(items), dtype=np.float32, format="csr")[items_to_idxs[item_id]].toarray()).to(device).float()

topk_elsa_scores, topk_elsa_idxs = elsa_with_sae.recommend_without_sae(interactions, k)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.flatten(), topk_elsa_idxs.flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations from ELSA:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
    .head(5)
)

topk_scores, topk_idxs, sparse_activations = elsa_with_sae.recommend_with_sae(interactions, k, activation_boost=activation_boost)
topk_scores, topk_idxs, sparse_activations = topk_scores.flatten(), topk_idxs.flatten(), sparse_activations.flatten()
topk_item_ids = np.array([items[iidx] for iidx in topk_idxs])
print("Recommendations from ELSA with SAE:")
print(
    pl.DataFrame({"item_id": topk_item_ids, "score": topk_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
    .head(10)
)

print(f"Active neuron count: {np.count_nonzero(sparse_activations)}")
topk_neuron_values, topk_neuron_ids = torch.topk(torch.tensor(sparse_activations), 10)
print(f"Most active neurons: {top_tag_per_neuron[topk_neuron_ids.cpu().numpy().flatten()]} with values {topk_neuron_values.detach().cpu().numpy().flatten()}")

User 152535
Interactions
shape: (11, 4)
┌─────────┬───────┬─────────────────────────────────┬──────────────────────────────┐
│ item_id ┆ value ┆ title                           ┆ genres                       │
│ ---     ┆ ---   ┆ ---                             ┆ ---                          │
│ cat     ┆ f32   ┆ str                             ┆ str                          │
╞═════════╪═══════╪═════════════════════════════════╪══════════════════════════════╡
│ 318     ┆ 4.5   ┆ Shawshank Redemption, The (199… ┆ Crime|Drama                  │
│ 785     ┆ 4.0   ┆ Kingpin (1996)                  ┆ Comedy                       │
│ 1242    ┆ 4.5   ┆ Glory (1989)                    ┆ Drama|War                    │
│ 1321    ┆ 4.0   ┆ American Werewolf in London, A… ┆ Comedy|Horror|Thriller       │
│ 2959    ┆ 4.0   ┆ Fight Club (1999)               ┆ Action|Crime|Drama|Thriller  │
│ …       ┆ …     ┆ …                               ┆ …                            │
│ 7360    ┆ 5.0   ┆ Dawn 

/tmp/ipykernel_1776867/2592974971.py:11: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  print(interactions_df.filter(pl.col("user_id") == user_id).join(items_df, on="item_id").select(["item_id", "value", "title", "genres"]))
/tmp/ipykernel_1776867/2592974971.py:23: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_1776867/2592974971.py:34: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
